---

# Extract.py

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # notebooks/ 의 상위 = 프로젝트 루트 (src 를 찾기 위함)

from src.extract import extract_pdf_elements, categorize_elements, PROJECT_ROOT, fpath, fname

In [ ]:
# 함수 실행 (hi_res 추출은 2~3분 소요)
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)
print("전체 요소:", len(raw_pdf_elements), "/ 텍스트:", len(texts), "/ 표:", len(tables))

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 57 65536 (offset 0)
Ignoring wrong pointing object 88 65536 (offset 0)
Ignoring wrong pointing object 119 65536 (offset 0)
Ignoring wrong pointing object 135 65536 (offset 0)
Ignoring wrong pointing object 138 65536 (offset 0)
Ignoring wrong pointing object 141 65536 (offset 0)
Ignoring wrong pointing object 144 65536 (offset 0)
Ignoring wrong pointing object 147 65536 (offset 0)
Ignoring wrong pointing object 182 65536 (offset 0)
Ignoring wrong pointing object 198 65536 (offset 0)
Ignoring wrong pointing object 208 65536 (offset 0)
Ignoring wrong pointing object 244 65536 (offset 0)
Ignoring wrong pointing object 253 65536 (offset 0)
Ignoring wr

In [ ]:
from collections import Counter

for etype, cnt in Counter(type(el).__name__ for el in raw_pdf_elements).most_common():
    print(f"{etype:20s}: {cnt}개")
figures_dir = Path(fpath) / "figures"
image_files = sorted(f for f in figures_dir.iterdir()
                        if f.suffix.lower() in (".jpg", ".jpeg", ".png"))
print("추출된 이미지:", len(image_files), "개")

In [ ]:
# 텍스트 재분할(토큰 기준 2000, 겹침 200) + extracted.pkl 저장  ← extract.py [6][7] 단계
import pickle
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=2000, chunk_overlap=200)
texts_2k_token = text_splitter.split_text(" ".join(texts))     # 블록을 합친 뒤 균일하게 재분할
print("분할 후 블록 수:", len(texts_2k_token))

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "extracted.pkl", "wb") as f:
    pickle.dump({"texts": texts, "tables": tables, "texts_2k_token": texts_2k_token}, f)
print("저장 완료:", output_dir / "extracted.pkl")

---

# Summarize.py

In [ ]:
import pickle
from dotenv import load_dotenv

from src.summarize import generate_text_summaries, generate_img_summaries

load_dotenv(PROJECT_ROOT / ".env")   # OPENAI_API_KEY 를 환경변수로 등록

In [ ]:
# 텍스트·표 요약 (LLM 호출, 비용 발생)
with open(PROJECT_ROOT / "data" / "processed" / "extracted.pkl", "rb") as f:
    data = pickle.load(f)

text_summaries, table_summaries = generate_text_summaries(
    data["texts_2k_token"], data["tables"], summarize_texts=True
)
print("텍스트 요약:", len(text_summaries), "/ 표 요약:", len(table_summaries))

In [ ]:
# 이미지 요약 (VLM 호출, 비용 발생)
img_base64_list, image_summaries = generate_img_summaries(str(PROJECT_ROOT / "data" / "figures"))
print("이미지 요약:", len(image_summaries), "장")

In [ ]:
# summaries.pkl 저장  ← summarize.py [8] 단계 (요약 재생성 비용 절약)
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "summaries.pkl", "wb") as f:
    pickle.dump({
        "text_summaries": text_summaries,     # 텍스트 요약 (검색용)
        "table_summaries": table_summaries,   # 표 요약 (검색용)
        "image_summaries": image_summaries,   # 이미지 요약 (검색용)
        "img_base64_list": img_base64_list,   # 이미지 원본 (답변용, 요약과 같은 순서)
    }, f)
print("저장 완료:", output_dir / "summaries.pkl")

# Retriever.py

In [ ]:
from src.retriever import load_processed, build_retriever, add_documents, build_chain, build_text_only_chain

In [ ]:
# extracted.pkl + summaries.pkl 복원 → 요약(서랍) + 원본(서가)을 같은 doc_id 로 연결해 색인 (임베딩 API 호출, 비용 발생)
# ⚠️ 이 셀은 실행할 때마다 컬렉션을 새로 비우고 다시 색인합니다 (중복 방지)
d1, d2 = load_processed()
tables_raw, texts_2k_token = d1["tables"], d1["texts_2k_token"]
text_summaries  = d2["text_summaries"]
table_summaries = d2["table_summaries"]
image_summaries = d2["image_summaries"]
img_base64_list = d2["img_base64_list"]

retriever = build_retriever()

if text_summaries:  add_documents(retriever, text_summaries,  texts_2k_token)
if table_summaries: add_documents(retriever, table_summaries, tables_raw)
if image_summaries: add_documents(retriever, image_summaries, img_base64_list)
print("색인된 요약 수:", len(retriever.vectorstore.get()["ids"]))

In [ ]:
# 멀티모달 RAG 질의 (LLM 호출, 비용 발생)
chain = build_chain(retriever)
query = "교보증권 제12879회 주가연계파생결합사채의 1증권당 발행가액과 최소청약금액은 얼마인가요?"   # ← 필요하면 다른 질문으로 교체
answer = chain.invoke(query)
print(f"Q: {query}\n{'-'*70}\n{answer}")

In [ ]:
# Before / After: 텍스트 전용 RAG 와 비교 (조건은 이미지·표 빼고 동일)
text_only_chain = build_text_only_chain(texts_2k_token)
print("[BEFORE] 텍스트 전용 RAG\n", text_only_chain.invoke(query))
print("[AFTER ] 멀티모달 RAG\n",   chain.invoke(query))

---

---